# CatBoost Optimization: Feature Engineering + Tuning

In [1]:
# Setup & load data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score

from src.preprocessing import PreprocessConfig, preprocess_train_test

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 250)

RANDOM_STATE = 42
TRAIN_PATH = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Train.csv"
TEST_PATH  = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

cfg = PreprocessConfig(id_col="ID", target_col="Target")
train, test = preprocess_train_test(train_raw, test_raw, cfg, for_model="catboost")

TARGET = cfg.target_col
ID = cfg.id_col

y = train[TARGET]
X = train.drop(columns=[TARGET])

In [2]:
# Make CatBoost-safe copies (categorical NaNs → "missing")

from pandas.api.types import is_numeric_dtype

cat_cols = [c for c in X.columns if not is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if is_numeric_dtype(X[c])]

X_cb = X.copy()
test_cb = test.copy()

for c in cat_cols:
    X_cb[c] = X_cb[c].astype("string").fillna("missing")
    test_cb[c] = test_cb[c].astype("string").fillna("missing")

# IMPORTANT: categorical indices for Pool
cat_feature_indices = [X_cb.columns.get_loc(c) for c in cat_cols]

print("cat:", len(cat_cols), "num:", len(num_cols))

cat: 32 num: 14


In [4]:
# Feature engineering function (train + test consistent)

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ---------
    # 1) Finance Access Score
    # ---------
    access_cols = [c for c in [
        "has_credit_card",
        "has_debit_card",
        "has_loan_account",
        "has_internet_banking",
        "has_mobile_money",
    ] if c in df.columns]

    # Convert yes/no-ish strings into 1/0 safely
    def yn_to01(s):
        s = s.astype("string").str.lower().fillna("missing")
        return s.isin(["yes", "y", "1", "true"]).astype(int)

    for c in access_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if access_cols:
        df["access_score"] = df[[f"{c}__01" for c in access_cols]].sum(axis=1)

    # ---------
    # 2) Insurance Coverage Score
    # ---------
    insurance_cols = [c for c in [
        "funeral_insurance",
        "medical_insurance",
        "motor_vehicle_insurance",
        "has_insurance",
    ] if c in df.columns]

    for c in insurance_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if insurance_cols:
        df["insurance_score"] = df[[f"{c}__01" for c in insurance_cols]].sum(axis=1)

    # ---------
    # 3) Resilience / Stress Score
    # ---------
    stress_cols = [c for c in [
        "current_problem_cash_flow",
        "attitude_worried_shutdown",
        "problem_sourcing_money",
    ] if c in df.columns]

    for c in stress_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if stress_cols:
        df["stress_score"] = df[[f"{c}__01" for c in stress_cols]].sum(axis=1)

    # ---------
    # 4) Financial Records / Compliance Score
    # ---------
    records_cols = [c for c in [
        "keeps_financial_records",
        "compliance_income_tax",
    ] if c in df.columns]

    for c in records_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if records_cols:
        df["formalization_score"] = df[[f"{c}__01" for c in records_cols]].sum(axis=1)

    # ---------
    # 5) Business scale ratios (only if numeric columns exist)
    # ---------
    if "business_turnover" in df.columns and "business_expenses" in df.columns:
        turn = pd.to_numeric(df["business_turnover"], errors="coerce")
        exp  = pd.to_numeric(df["business_expenses"], errors="coerce")
        df["turnover_minus_expenses"] = turn - exp
        df["turnover_to_expenses"] = turn / (exp.replace(0, np.nan))
        df["turnover_to_expenses"] = df["turnover_to_expenses"].replace([np.inf, -np.inf], np.nan)

    # ---------
    # 6) Age bucket (helps country/market nonlinearity)
    # ---------
    if "business_age_total_months" in df.columns:
        age = pd.to_numeric(df["business_age_total_months"], errors="coerce")
        df["business_age_years"] = (age / 12.0)
        df["age_bucket"] = pd.cut(
            df["business_age_years"],
            bins=[-np.inf, 0.5, 2, 5, 10, np.inf],
            labels=["<6m", "0.5-2y", "2-5y", "5-10y", "10y+"]
        ).astype("string")

    # ---------
    # Cleanup engineered ratio NaNs (CatBoost handles numeric NaNs OK)
    # ---------
    return df

In [5]:
# Apply it

X_fe = add_features(X_cb)
test_fe = add_features(test_cb)

# Recompute cat cols/indices after new features (important!)
from pandas.api.types import is_numeric_dtype
cat_cols_fe = [c for c in X_fe.columns if not is_numeric_dtype(X_fe[c])]
cat_feature_indices_fe = [X_fe.columns.get_loc(c) for c in cat_cols_fe]

print("Before:", X_cb.shape, "After FE:", X_fe.shape)

Before: (9618, 46) After FE: (9618, 67)
